In [1]:
import torch
import torch.nn as nn
import random

In [2]:
pairs = [
    ["hi", "hello"],
    ["how are you", "i am fine"],
    ["what is your name", "i am a chatbot"],
    ["what is ai", "ai is artificial intelligence"],
    ["bye", "goodbye"]
]


In [3]:
SOS_token = 0
EOS_token = 1

class Vocabulary:
    def __init__(self):
        self.word2index = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2

    def add_sentence(self, sentence):
        for word in sentence.split(" "):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

def tensor_from_sentence(vocab, sentence):
    indexes = [vocab.word2index[word] for word in sentence.split(" ")]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long)

# Build vocab
vocab = Vocabulary()
for pair in pairs:
    vocab.add_sentence(pair[0])
    vocab.add_sentence(pair[1])


In [4]:
hidden_size = 128

class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x).view(1, 1, -1)
        output, hidden = self.lstm(embedded, hidden)
        return output, hidden

    def init_hidden(self):
        return (torch.zeros(1, 1, self.hidden_size),
                torch.zeros(1, 1, self.hidden_size))

class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, hidden):
        embedded = self.embedding(x).view(1, 1, -1)
        output, hidden = self.lstm(embedded, hidden)
        output = self.softmax(self.fc(output[0]))
        return output, hidden

# Initialize models
encoder = Encoder(vocab.n_words, hidden_size)
decoder = Decoder(hidden_size, vocab.n_words)


In [5]:
encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr=0.01)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=0.01)
criterion = nn.NLLLoss()

def train_step(input_tensor, target_tensor):
    encoder_hidden = encoder.init_hidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    loss = 0

    # Encode
    for i in range(len(input_tensor)):
        _, encoder_hidden = encoder(input_tensor[i], encoder_hidden)

    decoder_input = torch.tensor([SOS_token])
    decoder_hidden = encoder_hidden

    # Decode
    for i in range(len(target_tensor)):
        output, decoder_hidden = decoder(decoder_input, decoder_hidden)
        loss += criterion(output, target_tensor[i].unsqueeze(0))
        decoder_input = target_tensor[i]

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / len(target_tensor)

print("Training started...")

for epoch in range(1000):
    total_loss = 0
    for pair in pairs:
        input_tensor = tensor_from_sentence(vocab, pair[0])
        target_tensor = tensor_from_sentence(vocab, pair[1])

        loss = train_step(input_tensor, target_tensor)
        total_loss += loss

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

print("Training complete!")

Training started...
Epoch 0, Loss: 15.0644
Epoch 200, Loss: 0.0004
Epoch 400, Loss: 0.0001
Epoch 600, Loss: 0.0000
Epoch 800, Loss: 0.0000
Training complete!


In [6]:
def evaluate(sentence):
    with torch.no_grad():
        try:
            input_tensor = tensor_from_sentence(vocab, sentence)
        except:
            return "I don't understand."

        hidden = encoder.init_hidden()

        for i in range(len(input_tensor)):
            _, hidden = encoder(input_tensor[i], hidden)

        decoder_input = torch.tensor([SOS_token])
        decoded_words = []

        for _ in range(10):
            output, hidden = decoder(decoder_input, hidden)
            topv, topi = output.topk(1)

            if topi.item() == EOS_token:
                break

            word = vocab.index2word.get(topi.item(), "")
            decoded_words.append(word)

            decoder_input = topi.squeeze().detach()

        return " ".join(decoded_words)

In [ ]:
print("\nChatbot ready! (type 'quit' to exit)\n")

while True:
    user_input = input("You: ").lower()

    if user_input == "quit":
        break

    response = evaluate(user_input)
    print("Bot:", response)


Chatbot ready! (type 'quit' to exit)

Bot: hello
Bot: i am a chatbot
Bot: i am fine
Bot: I don't understand.
Bot: I don't understand.


In [4]:
!pip install fastapi uvicorn


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from fastapi import FastAPI
print("FastAPI installed successfully")

FastAPI installed successfully


In [6]:
# model.py

# paste your FULL model code here (encoder, decoder, vocab, etc.)

def get_response(text):
    return evaluate(text)   # your chatbot function